In [ ]:
# ==============================
# MIGRATION INPUT PARAMETERS
# ==============================

analytical_linked_service = "< Add here >"
container_name = "< Add here >"

mirror_path = "< Add here >"
# e.g.: abfss://cdbnative@msit-onelake.dfs.fabric.microsoft.com/lh03.Lakehouse/Tables/container1

historical_data_write_path = "< Add here >"
# e.g.: "abfss://cdbnative@msit-onelake.dfs.fabric.microsoft.com/lh03.Lakehouse/Tables/container1_historical"

cutover_ts = 1735689600 # Unix timestamp


In [ ]:
# Step 1 Helper Functions (Schema Alignment Utility)

from pyspark.sql.functions import col, lit, to_json
from pyspark.sql.types import StructType, ArrayType, MapType, StringType

def align_to_mirror_schema(source_df, mirror_schema):
"""
Aligns source dataframe to mirror schema.
- Nested types converted to JSON when mirror expects string
- Type cast only if types differ
- Missing columns added as null
- Extra columns removed
"""

df = source_df

for field in mirror_schema:
    col_name = field.name
    if col_name in df.columns:
        source_field = df.schema[col_name]
        mirror_type = field.dataType
        source_type = source_field.dataType
        # Case 1: Nested JSON
        if isinstance(mirror_type, StringType) and isinstance(
            source_type, (StructType, ArrayType, MapType)
        ):
            df = df.withColumn(col_name, to_json(col(col_name)))
        # Case 2: Types differ -> cast
        elif source_type != mirror_type:
            df = df.withColumn(col_name, col(col_name).cast(mirror_type))
        # Case 3: Same type -> no-op
        else:
            pass
    else:
        df = df.withColumn(col_name, lit(None).cast(field.dataType))
# Reorder columns to match mirror schema
df = df.select([field.name for field in mirror_schema])

return df


In [ ]:
# Step 2 Read Mirror Schema

mirror_df = spark.read.format("delta").load(mirror_path)
mirror_schema = mirror_df.schema

print("Mirror Schema:")
mirror_df.printSchema()

In [ ]:
# Step 3 Read Analytical Store
analytical_df = (
spark.read
.format("cosmos.olap")
.option("spark.synapse.linkedService", analytical_linked_service)
.option("spark.cosmos.container", container_name)
.load()
)

print("Analytical Schema:")
analytical_df.printSchema()


In [ ]:
# Step 4 Align Schema

aligned_df = align_to_mirror_schema(analytical_df, mirror_schema)

print("Aligned Schema:")
aligned_df.printSchema()


In [ ]:
# Step 5 Apply Cutover Filter

historical_df = aligned_df.filter(col("_ts") < cutover_ts)

In [ ]:
# Step 6 Write Historical Dataset

(
historical_df.write
.format("delta")
.mode("overwrite")
.save(historical_data_write_path)
)
